In [1]:
### Definicja kolekcji hybrydowej

from qdrant_client import QdrantClient
from qdrant_client.http import models

client = QdrantClient(host="localhost", port=6333)
COLLECTION_NAME = "labor_code_pl"

# 1. Usuwa starą kolekcję
client.delete_collection(collection_name=COLLECTION_NAME)

# 2. Tworzy nową kolekcję z obsługą Dense i Sparse
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=1024, ### BGE-M3 ma 1024 wymiary
        distance=models.Distance.COSINE
    ),
    sparse_vectors_config={
        "text-sparse": models.SparseVectorParams() # miejsce na słowa kluczowe
    }
)

C:\Users\Greg Z\AppData\Local\Temp\ipykernel_22728\4272957500.py:6: UserWarning: Qdrant client version 1.13.0 is incompatible with server version 1.16.3. Major versions should match and minor version difference must not exceed 1. Set check_version=False to skip version check.
  client = QdrantClient(host="localhost", port=6333)


True

In [3]:
import re
from langchain_community.document_loaders import PyPDFLoader

# 1. Wczytanie dokumentu
file_path = r"O:\Projects\rag_labor_laws\backend\last_unified_labor_code.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

# 2. Łączenie stron i czyszczenie szumu (stopki, daty)
full_text = ""
for page in pages:
    content = page.page_content
    # Usuwanie stopki Kancelarii Sejmu i daty
    content = re.sub(r"©Kancelaria Sejmu.*s\.\s\d+/\d+", "", content)
    content = re.sub(r"2026-02-03", "", content) 
    full_text += content + "\n"

# 3. Funkcja dzieląca tekst na artykuły
def split_into_articles(text):
    # Szuka wzorca: "Art. [liczba]."
    pattern = r"(?=Art\.\s+\d+[a-z]*\.)"
    chunks = re.split(pattern, text)
    return [c.strip() for c in chunks if c.strip()]

articles = split_into_articles(full_text)

# 4. Przygotowanie finalnych obiektów do indeksowania
processed_data = []
for i, chunk in enumerate(articles):
    match = re.search(r"Art\.\s+(\d+[a-z]*)", chunk)
    art_number = f"Art. {match.group(1)}" if match else "Wstęp/Tytuł"
    
    processed_data.append({
        "content": chunk,
        "metadata": {
            "art_id": art_number,
            "source": "Kodeks Pracy",
            "status_date": "2026-02-03",
            "chunk_id": i
        }
    })

print(f"Przygotowano {len(processed_data)} artykułów.")
print(f"Pierwszy element: {processed_data[0]['metadata']['art_id']}")

Przygotowano 477 artykułów.
Pierwszy element: Wstęp/Tytuł


In [9]:
from fastembed import TextEmbedding

# Sprawdzenie jak wygląda struktura pojedynczego modelu
supported = TextEmbedding.list_supported_models()
if supported:
    print("Dostępne klucze w słowniku modelu:", supported[0].keys())
    for m in supported:
        print(m.get('model_name') or m.get('model') or m)

Dostępne klucze w słowniku modelu: dict_keys(['model', 'dim', 'description', 'license', 'size_in_GB', 'sources', 'model_file'])
BAAI/bge-base-en
BAAI/bge-base-en-v1.5
BAAI/bge-large-en-v1.5
BAAI/bge-small-en
BAAI/bge-small-en-v1.5
BAAI/bge-small-zh-v1.5
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
thenlper/gte-large
mixedbread-ai/mxbai-embed-large-v1
snowflake/snowflake-arctic-embed-xs
snowflake/snowflake-arctic-embed-s
snowflake/snowflake-arctic-embed-m
snowflake/snowflake-arctic-embed-m-long
snowflake/snowflake-arctic-embed-l
jinaai/jina-clip-v1
intfloat/multilingual-e5-large
sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Qdrant/clip-ViT-B-32-text
sentence-transformers/all-MiniLM-L6-v2
jinaai/jina-embeddings-v2-base-en
jinaai/jina-embeddings-v2-small-en
jinaai/jina-embeddings-v2-base-de
jinaai/jina-embeddings-v2-base-code
jinaai/jina-embeddings-v2-base-zh
jinaai/jina-embeddings-v2-base-es
thenlper/gte-base
nomic-ai/nomic-embed-text-v1.5
nomic-ai/nomic-embe

In [11]:
from fastembed import SparseTextEmbedding

sparse_supported = SparseTextEmbedding.list_supported_models()
for m in sparse_supported:
    model_name = m.get('model')
    if model_name and "m3" in model_name.lower():
        print(f"Poprawna nazwa to: {model_name}")

In [13]:
print("--- Dostępne modele SPARSE ---")
supported_sparse = SparseTextEmbedding.list_supported_models()
for m in supported_sparse:
    print(m.get('model'))

--- Dostępne modele SPARSE ---
prithivida/Splade_PP_en_v1
prithvida/Splade_PP_en_v1
Qdrant/bm42-all-minilm-l6-v2-attentions
Qdrant/bm25


In [15]:
from qdrant_client.http.models import PointStruct
import uuid

# 1. Inicjalizacja modeli BGE-M3 (Dense + Sparse)
# Model gęsty odpowiada za podobieństwo znaczeniowe (semantykę)
dense_model = TextEmbedding(model_name="intfloat/multilingual-e5-large")
# Model rzadki odpowiada za dokładne dopasowanie słów kluczowych i numerów artykułów
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

# 2. Przygotowanie tekstów do kodowania
documents = [item["content"] for item in processed_data]

# 3. Generowanie wektorów
print("Generowanie wektorów gęstych (dense)...")
dense_embeddings = list(dense_model.embed(documents))

print("Generowanie wektorów rzadkich (sparse)...")
sparse_embeddings = list(sparse_model.embed(documents))

# 4. Tworzenie punktów (PointStruct) dla bazy Qdrant
points = []
for i in range(len(processed_data)):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector={
                "": dense_embeddings[i].tolist(), # Główny wektor semantyczny
                "text-sparse": sparse_embeddings[i].as_object() # Wektor słów kluczowych
            },
            payload={
                "content": processed_data[i]["content"],
                "metadata": processed_data[i]["metadata"]
            }
        )
    )

print(f"Przygotowano {len(points)} hybrydowych punktów do wgrania.")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/716 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

O:\Projects\fashion_mnist\pyth311env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Greg Z\AppData\Local\Temp\fastembed_cache\models--qdrant--multilingual-e5-large-onnx. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

english.txt:   0%|          | 0.00/936 [00:00<?, ?B/s]

hinglish.txt: 0.00B [00:00, ?B/s]

hebrew.txt: 0.00B [00:00, ?B/s]

2026-03-11 23:54:39.667 | ERROR    | fastembed.common.model_management:download_model:274 - Could not download model from HuggingFace: [WinError 1314] Klient nie ma wymaganych uprawnień: '..\\..\\blobs\\27bf2940fe608d8a0698a7b896cd1fd22ad64ac9' -> 'C:\\Users\\GREGZ~1\\AppData\\Local\\Temp\\fastembed_cache\\models--Qdrant--bm25\\snapshots\\e499a1f8d6bec960aab5533a0941bf914e70faf9\\azerbaijani.txt' Falling back to other sources.
2026-03-11 23:54:39.667 | ERROR    | fastembed.common.model_management:download_model:295 - Could not download model from either source, sleeping for 3.0 seconds, 2 retries left.


Generowanie wektorów gęstych (dense)...
Generowanie wektorów rzadkich (sparse)...
Przygotowano 477 hybrydowych punktów do wgrania.
